# Improved GraphSAGE (v2)

Beats the 2026-08-20 GraphSAGE holdout (**ROC-AUC 0.767**, TP 3143, TN 67984, **FP 46060**).

This run uses **SIGN-style GraphSAGE**: past-only identity edges, then 1-hop and 2-hop *mean* neighbor features concatenated with the node itself, scored by an MLP. That is the same mean-aggregation as GraphSAGE, precomputed so CPU training finishes and we can verify TP/TN.

Also: milder `sqrt(n_neg/n_pos)` class weight, and an **MCC threshold** so true positives and true negatives are both high.

Kernel: `.gnn-venv`. Run All.

In [2]:
import gc
import json
import random
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy.sparse import csr_matrix
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler
from torch import nn

warnings.filterwarnings("ignore")

try:
    import google.colab
    IS_COLAB = True
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    IS_COLAB = False

ROOT = Path("/content/drive/MyDrive/minor-thesis") if IS_COLAB else Path.cwd()
DATASET_PATH = ROOT / "dataset"
SAVED_PATH = ROOT / "saved"
RESULTS_DIR = SAVED_PATH / "graph_models"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
K_NEIGHBORS = 8
HIDDEN = 128
DROPOUT = 0.2
EPOCHS = 40
PATIENCE = 8
BATCH = 8192
LR = 1e-3
MISSING_VALUES = {-999, -1}
MAX_GROUP_FRAC = 0.05
PREV = {"roc_auc": 0.7667, "tp": 3143, "tn": 67984, "fp": 46060, "fn": 921}

EDGE_KEYS = [
    {"name": "uid", "parts": ["uid"], "source": "engineered"},
    {"name": "uid2", "parts": ["uid2"], "source": "engineered"},
    {"name": "card1", "parts": ["card1"], "source": "single"},
    {"name": "link_card1_card2", "parts": ["card1", "card2"], "source": "pair"},
    {"name": "link_card1_addr1", "parts": ["card1", "addr1"], "source": "pair"},
    {"name": "link_uid_addr1", "parts": ["uid", "addr1"], "source": "pair"},
    {"name": "link_card1_R_emaildomain", "parts": ["card1", "R_emaildomain"], "source": "pair"},
]


def set_seed(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def select_node_features(columns):
    return [
        c
        for c in columns
        if c not in {"isFraud", "TransactionID"}
        and not c.startswith("id")
        and not c.startswith("V")
    ]


def key_series(df, parts):
    if len(parts) == 1:
        return df[parts[0]]
    cols = [df[p].to_numpy() for p in parts]
    return pd.Series(list(zip(*cols)), index=df.index)


def build_past_knn(df, parts, k=K_NEIGHBORS):
    keys = key_series(df, parts)
    invalid = pd.Series(False, index=df.index)
    for p in parts:
        invalid |= df[p].isin(MISSING_VALUES)
    rows, cols = [], []
    max_group = int(len(df) * MAX_GROUP_FRAC)
    work = df.loc[~invalid, ["TransactionDT"]].copy()
    work["_k"] = keys.loc[~invalid].to_numpy()
    for _, group in work.groupby("_k", sort=False):
        n = len(group)
        if n < 2 or n > max_group:
            continue
        idx = group.sort_values("TransactionDT").index.to_numpy(dtype=np.int64, copy=False)
        for offset in range(1, min(k + 1, n)):
            earlier, later = idx[:-offset], idx[offset:]
            rows.append(later)
            cols.append(earlier)
    if not rows:
        return np.zeros((2, 0), dtype=np.int64)
    return np.vstack([np.concatenate(rows), np.concatenate(cols)])


def coalesce_edges(edge_index, num_nodes):
    if edge_index.size == 0:
        return edge_index
    packed = edge_index[0].astype(np.int64) * np.int64(num_nodes) + edge_index[1].astype(np.int64)
    _, uniq = np.unique(packed, return_index=True)
    return edge_index[:, np.sort(uniq)]


def build_union_edges(df, edge_keys, k=K_NEIGHBORS):
    stats, chunks = [], []
    for spec in edge_keys:
        parts = [p for p in spec["parts"] if p in df.columns]
        if len(parts) != len(spec["parts"]):
            print(f"  skip {spec['name']}", flush=True)
            continue
        edges = build_past_knn(df, parts, k=k)
        stats.append({**spec, "directed_edges": int(edges.shape[1])})
        print(f"  {spec['name']:28s}  edges={edges.shape[1]:,}", flush=True)
        if edges.shape[1]:
            chunks.append(edges)
    union = (
        coalesce_edges(np.concatenate(chunks, axis=1), len(df))
        if chunks
        else np.zeros((2, 0), dtype=np.int64)
    )
    return union, stats


def sign_features(X, edge_index, num_nodes):
    """Precompute 1-hop and 2-hop mean neighbor features (SIGN / simplified SAGE)."""
    if edge_index.size == 0:
        zeros = np.zeros_like(X)
        deg = np.zeros((num_nodes, 1), dtype=np.float32)
        return np.hstack([X, zeros, zeros, deg])
    row, col = edge_index[0], edge_index[1]
    A = csr_matrix(
        (np.ones(row.shape[0], dtype=np.float32), (row, col)),
        shape=(num_nodes, num_nodes),
    )
    deg = np.asarray(A.sum(axis=1), dtype=np.float32).ravel()
    inv = np.reciprocal(np.clip(deg, 1.0, None))
    hop1 = inv[:, None] * (A @ X)
    hop2 = inv[:, None] * (A @ hop1)
    return np.hstack([X, hop1.astype(np.float32), hop2.astype(np.float32), np.log1p(deg)[:, None]])


class SignMLP(nn.Module):
    def __init__(self, in_channels, hidden=HIDDEN, dropout=DROPOUT):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_channels, hidden),
            nn.LayerNorm(hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def metrics_dict(name, y_true, y_prob, threshold, best_epoch=None):
    y_pred = (y_prob >= threshold).astype(np.int32)
    cm = confusion_matrix(y_true, y_pred)
    row = {
        "model": name,
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "pr_auc": float(average_precision_score(y_true, y_prob)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "tn": int(cm[0, 0]),
        "fp": int(cm[0, 1]),
        "fn": int(cm[1, 0]),
        "tp": int(cm[1, 1]),
    }
    if best_epoch is not None:
        row["best_epoch"] = int(best_epoch)
    return row


def tune_threshold(y_true, y_prob, min_recall=0.30):
    best_mcc, best_thr = -1.0, 0.5
    for thr in np.linspace(0.05, 0.95, 91):
        pred = (y_prob >= thr).astype(np.int32)
        if recall_score(y_true, pred, zero_division=0) < min_recall:
            continue
        mcc = matthews_corrcoef(y_true, pred)
        if mcc > best_mcc:
            best_mcc, best_thr = mcc, float(thr)
    return best_thr


set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, flush=True)

train = pd.read_parquet(DATASET_PATH / "merged_train.parquet")
train = train.sort_values("TransactionDT").reset_index(drop=True)
print(f"rows={len(train):,} fraud={train['isFraud'].mean():.4f}", flush=True)

feature_cols = select_node_features(list(train.columns))
y = train["isFraud"].to_numpy(dtype=np.int64)
split = int(len(train) * 0.8)
train_idx = np.arange(0, split)
val_idx = np.arange(split, len(train))

scaler = StandardScaler()
X = train[feature_cols].to_numpy(dtype=np.float32)
X[train_idx] = scaler.fit_transform(X[train_idx])
X[val_idx] = scaler.transform(X[val_idx])

print("Building past-only identity edges...", flush=True)
edge_index, edge_stats = build_union_edges(train, EDGE_KEYS, k=K_NEIGHBORS)
print(f"union directed edges: {edge_index.shape[1]:,}", flush=True)
deg = np.bincount(edge_index[0], minlength=len(train)) if edge_index.size else np.zeros(len(train))
print(f"avg in-degree {deg.mean():.2f}  isolated {(deg == 0).sum():,}", flush=True)

print("Precomputing 1-hop / 2-hop mean features...", flush=True)
Z = sign_features(X, edge_index, len(train))
print(f"SIGN feature dim={Z.shape[1]}", flush=True)
del X, edge_index
gc.collect()

n_pos = max(int(y[train_idx].sum()), 1)
n_neg = len(train_idx) - n_pos
pos_weight = torch.tensor([float(np.sqrt(n_neg / n_pos))], dtype=torch.float32)
print(f"pos_weight={float(pos_weight):.2f}", flush=True)

z_all = torch.from_numpy(Z)
y_t = torch.from_numpy(y.astype(np.float32))
model = SignMLP(Z.shape[1]).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-5)
crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

history = []
best_auc, best_epoch, stalled = -1.0, 0, 0
best_state = None
rng = np.random.default_rng(RANDOM_SEED)

for epoch in range(1, EPOCHS + 1):
    model.train()
    perm = rng.permutation(train_idx)
    total = 0.0
    nbat = 0
    for start in range(0, len(perm), BATCH):
        b = perm[start : start + BATCH]
        xb = z_all[b].to(device)
        yb = y_t[b].to(device)
        opt.zero_grad()
        loss = crit(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
        total += float(loss.item())
        nbat += 1
    sched.step()
    model.eval()
    with torch.no_grad():
        logits = []
        for start in range(0, len(val_idx), BATCH):
            b = val_idx[start : start + BATCH]
            logits.append(model(z_all[b].to(device)).cpu())
        y_prob = torch.sigmoid(torch.cat(logits)).numpy()
    auc = roc_auc_score(y[val_idx], y_prob)
    prauc = average_precision_score(y[val_idx], y_prob)
    avg = total / max(nbat, 1)
    history.append({"epoch": epoch, "loss": avg, "roc_auc": float(auc), "pr_auc": float(prauc)})
    print(f"epoch {epoch:02d}/{EPOCHS} loss {avg:.4f} ROC-AUC {auc:.4f} PR-AUC {prauc:.4f}", flush=True)
    if auc > best_auc + 1e-4:
        best_auc, best_epoch, stalled = auc, epoch, 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        stalled += 1
        if stalled >= PATIENCE:
            print(f"early stop at {epoch} best {best_auc:.4f} @{best_epoch}", flush=True)
            break

if best_state is not None:
    model.load_state_dict(best_state)
model.eval()
with torch.no_grad():
    logits = []
    for start in range(0, len(val_idx), BATCH):
        b = val_idx[start : start + BATCH]
        logits.append(model(z_all[b].to(device)).cpu())
    y_prob = torch.sigmoid(torch.cat(logits)).numpy()
y_true = y[val_idx]

thr = tune_threshold(y_true, y_prob, min_recall=0.30)
at_half = metrics_dict("GraphSAGE-SIGN-v2 @0.5", y_true, y_prob, 0.5, best_epoch)
at_mcc = metrics_dict("GraphSAGE-SIGN-v2 MCC-thr", y_true, y_prob, thr, best_epoch)

print(f"\nChosen MCC threshold: {thr:.2f}")
print(f"Previous GraphSAGE: ROC-AUC {PREV['roc_auc']:.4f} TP {PREV['tp']} TN {PREV['tn']} FP {PREV['fp']}")
for row in (at_half, at_mcc):
    print(f"\n{row['model']}")
    for k in ["threshold", "roc_auc", "pr_auc", "f1", "mcc", "recall", "precision", "tn", "fp", "fn", "tp"]:
        print(f"  {k}: {row[k]}")
    print(
        classification_report(
            y_true,
            (y_prob >= row["threshold"]).astype(int),
            target_names=["Legitimate", "Fraud"],
            digits=4,
            zero_division=0,
        )
    )

print(f"ROC-AUC delta: {at_mcc['roc_auc'] - PREV['roc_auc']:+.4f}")
print(f"TP delta: {at_mcc['tp'] - PREV['tp']:+d}  TN delta: {at_mcc['tn'] - PREV['tn']:+d}")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
torch.save(
    {"state_dict": model.state_dict(), "in_channels": Z.shape[1], "feature_cols": feature_cols},
    RESULTS_DIR / f"graphsage_v2_{timestamp}.pt",
)
payload = {
    "timestamp": timestamp,
    "variant": "GraphSAGE-SIGN-v2",
    "k_neighbors": K_NEIGHBORS,
    "hidden": HIDDEN,
    "edge_keys": EDGE_KEYS,
    "edge_stats": edge_stats,
    "pos_weight": float(pos_weight),
    "best_epoch": int(best_epoch),
    "threshold_mcc": float(thr),
    "previous": PREV,
    "metrics_0.5": at_half,
    "metrics_mcc_threshold": at_mcc,
    "history": history,
    "node_features": feature_cols,
}
with open(RESULTS_DIR / f"gnn_v2_metadata_{timestamp}.json", "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2)
pd.DataFrame([at_half, at_mcc]).to_csv(RESULTS_DIR / f"gnn_v2_results_{timestamp}.csv", index=False)
print("saved", RESULTS_DIR / f"graphsage_v2_{timestamp}.pt", flush=True)

Device: cpu
rows=590,540 fraud=0.0350
Building past-only identity edges...
  uid                           edges=4,396,626
  uid2                          edges=2,159,334
  card1                         edges=4,398,625
  link_card1_card2              edges=4,328,822
  link_card1_addr1              edges=3,454,801
  link_uid_addr1                edges=3,441,563
  link_card1_R_emaildomain      edges=4,183,074
union directed edges: 8,408,088
avg in-degree 14.24  isolated 13,088
Precomputing 1-hop / 2-hop mean features...
SIGN feature dim=187
pos_weight=5.24
epoch 01/40 loss 0.4499 ROC-AUC 0.7957 PR-AUC 0.2326
epoch 02/40 loss 0.3816 ROC-AUC 0.8165 PR-AUC 0.2798
epoch 03/40 loss 0.3623 ROC-AUC 0.8258 PR-AUC 0.2963
epoch 04/40 loss 0.3518 ROC-AUC 0.8283 PR-AUC 0.3047
epoch 05/40 loss 0.3422 ROC-AUC 0.8321 PR-AUC 0.3038
epoch 06/40 loss 0.3356 ROC-AUC 0.8338 PR-AUC 0.3178
epoch 07/40 loss 0.3293 ROC-AUC 0.8369 PR-AUC 0.3252
epoch 08/40 loss 0.3240 ROC-AUC 0.8329 PR-AUC 0.3155
epoch 09/40 los